# 02 - Price spreads and net-price opportunities (Q1, Q2)

Q1: how big is the same-day price gap between nearby mandis (<= 100 km road estimate)?  
Q2: after transport costs, how often is selling at a farther mandi profitable, and by how much?

In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from mandipulse import viz
from mandipulse.viz import COMMODITIES, COMMODITY_COLORS, INK, INK_2, MUTED, load
pd.set_option("display.width", 160)
viz.style()

## Q1 - same-day gap between nearby markets

In [2]:
q1 = load("phase3", "q1_spread_by_commodity")
q1

,commodity,n_pair_days,n_pairs,median_abs_gap_rs_qtl,p90_abs_gap_rs_qtl,median_pct_gap,p90_pct_gap,pct_pair_days_gap_ge_20pct
0,Onion,2331040,3220,150.0,675.0,10.5,55.0,33.1
1,Potato,2419690,2288,100.0,450.0,11.1,56.3,35.5
2,Tomato,2026802,2262,200.0,830.0,15.3,77.4,43.2


In [3]:
dist = load("phase3", "q1_gap_by_distance")
dist["band_mid"] = dist["road_km_band_start"] + 10
fig, ax = viz.figure("Q1. Same-day price gap between market pairs, by distance",
                     "Median gap as % of the lower price, pairs <= 100 km road estimate, days both reported")
for c in COMMODITIES:
    g = dist[dist["commodity"] == c]
    ax.plot(g["band_mid"], g["median_pct_gap"], color=COMMODITY_COLORS[c], marker="o", markersize=5, label=c)
viz.label_line_ends(ax, dist, "band_mid", "median_pct_gap", "commodity", fmt="{:.1f}%")
ax.set_xlabel("road distance between the two markets, km (20 km bands)")
ax.set_ylabel("median gap, % of lower price")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "q1_gap_by_distance", note="Road distance = straight-line distance x 1.3 (assumption).")
dist.pivot(index="road_km_band_start", columns="commodity", values="median_pct_gap")

commodity,Onion,Potato,Tomato
road_km_band_start,,,
0,9.9,10.1,14.3
20,10.5,11.5,15.3
40,9.5,10.0,14.3
60,10.7,11.4,15.3
80,11.1,11.7,16.0


In [4]:
mon = load("phase3", "q1_spread_monthly")
mon["month"] = pd.to_datetime(mon["month"])
fig, ax = viz.figure("Q1. Same-day gap between nearby markets over time",
                     "Monthly median gap as % of the lower price, pairs <= 100 km")
for c in COMMODITIES:
    g = mon[mon["commodity"] == c]
    ax.plot(g["month"], g["median_pct_gap"], color=COMMODITY_COLORS[c], label=c)
viz.label_line_ends(ax, mon, "month", "median_pct_gap", "commodity", fmt="{:.1f}%")
ax.set_ylabel("median gap, %")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "q1_gap_monthly")
mon.groupby("commodity")["median_pct_gap"].describe()

,count,mean,std,min,25%,50%,75%,max
commodity,,,,,,,,
Onion,94.0,11.035106,3.835480,5.4,8.300,10.0,13.275,20.6
Potato,94.0,11.634043,4.593229,4.7,8.025,10.4,14.900,21.3
Tomato,94.0,15.855319,5.934637,5.6,10.875,15.3,18.875,32.0


## Q2 - profitable moves after transport cost

In [5]:
q2 = load("phase3", "q2_opportunity_rate")
scen = ["low", "mid", "high"]
fig, ax = viz.figure("Q2. Market-days with a profitable nearby market, after transport and fees",
                     "% of market-days where >= 1 market within 100 km pays >= Rs 100/qtl and >= 5% more, net of transport and fees")
x = np.arange(len(scen))
w = 0.26
for i, c in enumerate(COMMODITIES):
    g = q2[q2["commodity"] == c].set_index("scenario").loc[scen]
    bars = ax.bar(x + (i - 1) * (w + 0.01), g["pct_market_days_with_opportunity"], width=w,
                  color=COMMODITY_COLORS[c], label=c)
    for b, v in zip(bars, g["pct_market_days_with_opportunity"], strict=True):
        ax.annotate(f"{v:.0f}%", (b.get_x() + b.get_width() / 2, v), xytext=(0, 3),
                    textcoords="offset points", ha="center", fontsize=8.5, color=INK_2)
ax.set_xticks(x, ["low cost\n(Rs 1.0/qtl/km + 30, 4% fees)", "mid cost\n(Rs 1.5/qtl/km + 50, 6% fees)", "high cost\n(Rs 2.5/qtl/km + 80, 8% fees)"])
ax.set_ylabel("% of market-days")
ax.set_ylim(0, 70)
viz.legend(ax)
viz.save(fig, "q2_opportunity_rate_by_scenario",
         note="~8 neighbours report per day; any single one is profitable on only 5-20% of comparisons (see table). "
              "Fees 4/6/8%; spoilage not modelled.")
q2

,scenario,commodity,n_market_days,pct_market_days_with_opportunity,median_best_gain_rs_qtl,avg_neighbors_reporting,pct_single_comparisons_opportunity
0,low,Onion,580351,42.2,370.0,8.0,16.1
1,mid,Onion,580351,33.4,372.0,8.0,12.0
2,high,Onion,580351,23.6,385.0,8.0,7.8
3,low,Potato,537622,32.9,284.0,9.0,12.9
4,mid,Potato,537622,24.9,278.0,9.0,9.0
5,high,Potato,537622,16.3,270.0,9.0,5.1
6,low,Tomato,542037,51.4,456.0,7.5,19.8
7,mid,Tomato,542037,43.2,451.0,7.5,15.4
8,high,Tomato,542037,33.2,442.0,7.5,10.8


In [6]:
gd = load("phase3", "q2_gain_distribution")
fig, ax = viz.figure("Q2. Size of the best gain on opportunity days (mid cost)",
                     "Share of opportunity market-days by best net gain, Rs per quintal (bins of 100; last bin = 2000+)")
for c in COMMODITIES:
    g = gd[gd["commodity"] == c].copy()
    g["share"] = 100 * g["n_market_days"] / g["n_market_days"].sum()
    ax.step(g["gain_bin_start_rs_qtl"], g["share"], where="post", color=COMMODITY_COLORS[c], label=c)
ax.set_xlabel("best net gain, Rs / quintal")
ax.set_ylabel("% of opportunity market-days")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "q2_gain_distribution")
gd.pivot(index="gain_bin_start_rs_qtl", columns="commodity", values="n_market_days").head(8)

commodity,Onion,Potato,Tomato
gain_bin_start_rs_qtl,,,
100,43471,42560,40421
200,34705,30186,35613
300,24877,19959,28580
400,18633,12700,23528
500,14537,8625,19587
600,11096,6005,16024
700,9102,4478,12689
800,7469,2969,10258


In [7]:
yr = load("phase3", "q2_opportunity_by_year")
yr.pivot(index="year", columns="commodity", values="pct_market_days_with_opportunity")

commodity,Onion,Potato,Tomato
year,,,
2018,34.9,27.8,42.8
2019,36.3,26.2,53.7
2020,33.0,21.9,41.0
2021,34.2,21.7,41.3
2022,34.8,25.7,47.4
2023,28.6,21.0,39.0
2024,30.3,26.0,39.2
2025,35.7,29.7,39.6


## Sanity checks behind the Q2 number

In [8]:
for q in ["sanity_02_opportunity_concentration", "sanity_04_structural_vs_sporadic",
          "sanity_05_actionable_opportunities", "sanity_06_variety_mix", "sanity_07_opportunity_by_state"]:
    print(q)
    display(load("phase3", q))

sanity_02_opportunity_concentration


,slice,pct_of_opportunities
0,top 10 destination markets,14.8
1,top 5% of destination markets,30.7
2,top 20% of destination markets,67.0


sanity_04_structural_vs_sporadic


,pair_profitable_on,n_directed_pairs,comparison_days,opportunity_days,pct_of_all_opportunity_days
0,0-20% of days,11174,10767663.0,579163.0,35.8
1,20-40% of days,2330,1780292.0,505072.0,31.2
2,40-60% of days,1210,777520.0,375967.0,23.2
3,60-80% of days,474,212830.0,142905.0,8.8
4,80-100% of days,352,16759.0,14575.0,0.9


sanity_05_actionable_opportunities


,commodity,n_signals,pct_signal_still_opportunity,pct_signal_gain_positive,median_realised_gain_after_signal,median_gain_ex_post
0,Onion,481574,79.3,88.4,281.0,322.0
1,Potato,397167,84.2,93.3,225.0,242.0
2,Tomato,561645,79.5,87.9,314.0,359.0


sanity_06_variety_mix


,pair_type,commodity,n_comparisons,pct_opportunity,median_pct_gap
0,different dominant variety,Onion,903458,19.4,0.194
1,same dominant variety,Onion,3758622,10.2,0.085
2,different dominant variety,Potato,2549820,10.3,0.128
3,same dominant variety,Potato,2289560,7.5,0.086
4,different dominant variety,Tomato,2189384,15.6,0.143
5,same dominant variety,Tomato,1864220,15.2,0.139


sanity_07_opportunity_by_state


,state,commodity,n_market_days,pct_of_commodity_market_days,pct_with_opportunity,avg_neighbors
0,Gujarat,Onion,28945,5.0,26.2,2.4
1,Madhya Pradesh,Onion,63937,11.0,46.3,4.3
2,Maharashtra,Onion,95133,16.4,32.3,7.2
3,Uttar Pradesh,Onion,392336,67.6,32.1,9.3
4,Gujarat,Potato,25827,4.8,31.8,3.2
5,Madhya Pradesh,Potato,45173,8.4,40.1,3.2
6,Maharashtra,Potato,34994,6.5,31.1,2.3
7,Uttar Pradesh,Potato,431628,80.3,22.4,10.5
8,Gujarat,Tomato,40121,7.4,40.2,3.2
9,Madhya Pradesh,Tomato,48111,8.9,48.7,3.1
